In [29]:
import pandas as pd

df = pd.read_csv("2010_data.csv", low_memory=False)
df.columns = df.columns.str.lower()

# Select the first trading day of 2010
df = df[df["dlycaldt"] == "2010-01-04"]

# Filter US common stocks
df = df[
    (df["securitytype"] == "EQTY") &
    (df["securitysubtype"] == "COM") &
    (df["sharetype"] == "NS") &
    (df["usincflg"] == "Y") &
    (df["issuertype"].isin(["ACOR", "CORP"]))
]

# Remove duplicates and invalid market caps
df = df.drop_duplicates(subset="permno")
df = df[df["dlycap"] > 0]

# Rank the top 100 companies by market cap
top100 = df.groupby("permco").agg(
    company_name=("issuernm", "first"),
    market_cap=("dlycap", "sum")
).nlargest(100, "market_cap")

# Display the top 100 companies
print(top100.to_string())

# Export company list
top100.to_csv("2010_Top100_US_Equities.csv")

# Print PERMCOs
print(" ".join(top100.index.astype(int).astype(str)))

                            company_name    market_cap
permco                                                
20678                   EXXON MOBIL CORP  3.272107e+08
8048                      MICROSOFT CORP  2.727004e+08
21880                WAL MART STORES INC  2.066256e+08
7                              APPLE INC  1.939534e+08
21446                PROCTER & GAMBLE CO  1.785764e+08
21018                  JOHNSON & JOHNSON  1.784586e+08
20436                JPMORGAN CHASE & CO  1.758964e+08
20990   INTERNATIONAL BUSINESS MACHS COR  1.739867e+08
21645                        A T & T INC  1.686506e+08
20792                GENERAL ELECTRIC CO  1.645038e+08
20440                   CHEVRON CORP NEW  1.586155e+08
540           BERKSHIRE HATHAWAY INC DEL  1.544055e+08
21394                         PFIZER INC  1.527563e+08
45483                         GOOGLE INC  1.526832e+08
10486                  CISCO SYSTEMS INC  1.420313e+08
21305               WELLS FARGO & CO NEW  1.396825e+08
3151      

In [30]:
#data cleaning
import pandas as pd

df = pd.read_csv("Top100_2010-2025.csv", low_memory=False)

# Standardise column names
df.columns = df.columns.str.lower()

# Convert dates
df["dlycaldt"] = pd.to_datetime(df["dlycaldt"])

# Remove exact duplicates
df = df.drop_duplicates()

# Sort the data
df = df.sort_values(["permco", "permno", "dlycaldt"])

# Check cleaned data
print(df.shape)
print("Number of companies:", df["permco"].nunique())
print("Missing returns:", df["dlyret"].isna().sum())

(379099, 53)
Number of companies: 100
Missing returns: 1


In [31]:
#there is a missing return
df[df["dlyret"].isna()][
    ["permco", "permno", "ticker", "issuernm",
     "dlycaldt", "dlyret", "dlyretmissflg"]
]

,permco,permno,ticker,issuernm,dlycaldt,dlyret,dlyretmissflg
59388,45483,14542,GOOG,GOOGLE INC,2014-04-03,NaN,NS


In [34]:
#there are 4 more permcos
multiple = df.groupby("permco")["permno"].nunique()

print(multiple[multiple > 1])

permco
540      2
21287    2
43613    2
45483    2
Name: permno, dtype: int64


In [36]:
#fixing the multiple permcos
multiple_ids = [540, 21287, 43613, 45483]

print(
    df[df["permco"].isin(multiple_ids)]
    [["permco", "permno", "ticker", "issuernm", "shareclass"]]
    .drop_duplicates()
    .to_string(index=False)
)

df[df["permco"].isin(multiple_ids)].groupby(
    ["permco", "permno"]
).agg(
    company_name=("issuernm", "first"),
    ticker=("ticker", "first"),
    start_date=("dlycaldt", "min"),
    end_date=("dlycaldt", "max")
)

 permco  permno ticker                                  issuernm shareclass
    540   17778    BRK                BERKSHIRE HATHAWAY INC DEL          A
    540   83443    BRK                BERKSHIRE HATHAWAY INC DEL          B
  21287   90441   NWSA                                 NEWS CORP          A
  21287   90441   FOXA              TWENTY FIRST CENTURY FOX INC          A
  21287   90441  TFCFA              TWENTY FIRST CENTURY FOX INC          A
  21287   90441    NaN TWENTY FIRST CENTURY FOX INC (Last Known)        NaN
  21287   90442    NWS                                 NEWS CORP          B
  21287   90442    FOX              TWENTY FIRST CENTURY FOX INC          B
  21287   90442   TFCF              TWENTY FIRST CENTURY FOX INC          B
  21287   90442    NaN TWENTY FIRST CENTURY FOX INC (Last Known)        NaN
  43613   89525  CMCSA                          COMCAST CORP NEW          A
  43613   89565  CMCSK                          COMCAST CORP NEW        NaN
  43613   89

company_name ticker start_date   end_date
permco permno                                                         
540    17778   BERKSHIRE HATHAWAY INC DEL    BRK 2010-01-04 2025-12-31
       83443   BERKSHIRE HATHAWAY INC DEL    BRK 2010-01-04 2025-12-31
21287  90441                    NEWS CORP   NWSA 2010-01-04 2019-03-20
       90442                    NEWS CORP    NWS 2010-01-04 2019-03-20
43613  89525             COMCAST CORP NEW  CMCSA 2010-01-04 2025-12-31
       89565             COMCAST CORP NEW  CMCSK 2010-01-04 2015-12-14
45483  14542                   GOOGLE INC   GOOG 2014-04-03 2025-12-31
       90319                   GOOGLE INC   GOOG 2010-01-04 2025-12-31

In [37]:
# Handle multiple share classes using market-cap-weighted returns (将同一家公司的不同 Share Classes 看作一个小型投资组合，根据各个 Share Class 上一个交易日的市值占比进行加权，计算整个公司的 Daily Return)
# QMIM Portfolio Theory, Slides 7-8:
# Portfolio return = sum(weight * asset return)
# Use previous-day market cap to avoid look-ahead bias

# Keep valid returns and previous market capitalisations
valid = df.dropna(subset=["dlyret", "dlyprevcap"]).copy()
valid = valid[valid["dlyprevcap"] > 0]

# Calculate weighted returns
valid["weighted_ret"] = valid["dlyret"] * valid["dlyprevcap"]

# Aggregate multiple share classes at company level
company_returns = valid.groupby(["dlycaldt", "permco"])[
    ["weighted_ret", "dlyprevcap"]
].sum()

# Calculate company-level daily returns
company_returns["return"] = (
    company_returns["weighted_ret"] /
    company_returns["dlyprevcap"]
)

company_returns = company_returns.reset_index()

company_returns.head()

,dlycaldt,permco,weighted_ret,dlyprevcap,return
0,2010-01-04,7,2.970735e+06,1.909826e+08,0.015555
1,2010-01-04,90,4.756836e+05,4.818513e+07,0.009872
2,2010-01-04,216,1.139590e+06,5.628715e+07,0.020246
3,2010-01-04,540,7.940713e+05,1.536114e+08,0.005169
4,2010-01-04,1645,7.075909e+05,4.304866e+07,0.016437


In [38]:
#check
print("Number of companies:", company_returns["permco"].nunique())

print("Duplicate records:",
      company_returns.duplicated(["dlycaldt", "permco"]).sum())

print("Missing returns:", company_returns["return"].isna().sum())

Number of companies: 100
Duplicate records: 0
Missing returns: 0


In [39]:
#construct daily return matrix
return_matrix = company_returns.pivot(
    index="dlycaldt",
    columns="permco",
    values="return"
).sort_index()

print("Return matrix shape:", return_matrix.shape)

return_matrix.head()

Return matrix shape: (4024, 100)


permco,7,90,216,540,1645,1685,2367,2709,2850,3151,...,41680,42273,43613,44277,44708,45483,45695,50700,52978,52983
dlycaldt,,,,,,,,,,,,,,,,,,,,,
2010-01-04,0.015555,0.009872,0.020246,0.005169,0.016437,0.000000,0.023529,-0.009833,-0.001819,0.041833,...,0.009198,0.025047,0.011320,0.018776,0.011694,0.010920,0.033149,0.003360,0.024071,0.007775
2010-01-05,0.001729,-0.002199,-0.008582,0.002293,0.025350,0.013062,-0.000479,-0.010363,0.020501,0.032505,...,0.048852,0.010879,-0.011972,-0.018738,0.005335,-0.004404,0.023335,-0.002959,-0.008713,-0.011459
2010-01-06,-0.015906,0.016165,-0.007515,-0.000678,0.010656,-0.008280,-0.003354,0.002618,0.016406,0.011728,...,0.006952,0.016225,-0.006771,-0.004383,-0.014741,-0.025209,0.038321,-0.001445,0.000000,-0.013428
2010-01-07,-0.001849,0.016148,-0.009157,0.002269,0.019401,-0.010854,-0.009615,0.028721,0.009265,0.032947,...,-0.002416,0.037146,0.021491,0.011162,0.013465,-0.023280,0.002898,-0.006535,0.012469,0.009307
2010-01-08,0.006648,-0.000715,0.008886,0.001953,0.001655,0.024840,0.011165,-0.002115,0.005246,-0.008860,...,0.001038,0.004398,-0.003144,0.011350,0.007381,0.013331,0.014294,0.000354,-0.005451,0.002766


In [40]:
# Check missing returns by company
missing = return_matrix.isna().sum()

print(missing[missing > 0].sort_values(ascending=False))

permco
20350    3994
29842    3902
44277    3457
9833     3060
44708    2624
8093     2342
20597    2093
20606    2093
40148    1902
11412    1896
29540    1800
21287    1706
20194    1607
9374     1534
21881      86
dtype: int64


In [43]:
# Identify companies with missing returns
missing_ids = missing[missing > 0].index

# Check company names and data coverage
summary = df[df["permco"].isin(missing_ids)].groupby("permco").agg(
    company_name=("issuernm", "first"),
    start_date=("dlycaldt", "min"),
    end_date=("dlycaldt", "max")
)

summary["missing_days"] = missing

summary.sort_values("missing_days", ascending=False)
#these are the companies that quit the market

,company_name,start_date,end_date,missing_days
permco,,,,
20350,BURLINGTON NORTHERN SANTA FE CP,2010-01-04,2010-02-16,3994
29842,X T O ENERGY INC,2010-01-04,2010-06-28,3902
44277,MEDCO HEALTH SOLUTIONS INC,2010-01-04,2012-04-02,3457
9833,DELL INC,2010-01-04,2013-10-30,3060
44708,DIRECTV,2010-01-04,2015-07-27,2624
8093,E M C CORP MA,2010-01-04,2016-09-07,2342
20597,DOW CHEMICAL CO,2010-01-04,2017-09-01,2093
20606,DU PONT E I DE NEMOURS & CO,2010-01-04,2017-09-01,2093
40148,MONSANTO CO NEW,2010-01-04,2018-06-07,1902


In [44]:
# Check missing returns within each company's observed return history
internal_missing = return_matrix.apply(
    lambda x: x.loc[
        x.first_valid_index():x.last_valid_index()
    ].isna().sum()
)

print(internal_missing[internal_missing > 0])

Series([], dtype: int64)


In [45]:
# Check the final observation of companies with missing returns
last_records = (
    df[df["permco"].isin(missing[missing > 0].index)]
    .sort_values("dlycaldt")
    .groupby("permco")
    .tail(1)
)

last_records[[
    "permco", "issuernm", "dlycaldt",
    "dlyret", "dlydelflg", "delreasontype"
]]

,permco,issuernm,dlycaldt,dlyret,dlydelflg,delreasontype
205065,20350,BURLINGTON NORTHERN SANTA FE CP (Last Known),2010-02-16,-0.000337,Y,NaN
292440,29842,X T O ENERGY INC (Last Known),2010-06-28,-0.007366,Y,NaN
348968,44277,MEDCO HEALTH SOLUTIONS INC (Last Known),2012-04-02,0.049147,Y,NaN
14720,9833,DELL INC (Last Known),2013-10-30,0.001443,Y,NaN
350369,44708,DIRECTV (Last Known),2015-07-27,-0.001251,Y,NaN
13755,8093,E M C CORP MA (Last Known),2016-09-07,0.012051,Y,NaN
120525,20597,DOW CHEMICAL CO (Last Known),2017-09-01,0.007952,Y,NaN
23168,20606,DU PONT E I DE NEMOURS & CO (Last Known),2017-09-01,0.026150,Y,NaN
334831,40148,MONSANTO CO NEW (Last Known),2018-06-07,0.000391,Y,NaN
290093,11412,TIME WARNER INC NEW (Last Known),2018-06-15,0.026491,Y,NaN


In [47]:
# Check all delisting records
delisted = df[df["dlydelflg"] == "Y"]

print("Delisting records:", len(delisted))
print("Missing delisting returns:", delisted["dlyret"].isna().sum())
print("Missing previous market cap:", delisted["dlyprevcap"].isna().sum())
print("Non-positive previous market cap:",
      (delisted["dlyprevcap"] <= 0).sum())

Delisting records: 17
Missing delisting returns: 0
Missing previous market cap: 0
Non-positive previous market cap: 0


Delisting and Missing Returns. After aggregating multiple share classes, we identified 15 companies with missing daily returns. However, no missing returns were found between each company's first and last valid return dates.
We therefore check the delisting records to determine whether these missing values are related to companies exiting the investment universe.
Following QMIM Signal Construction and Testing (Slide 7), we retain delisted stocks and their terminal returns to avoid survivorship bias.
We identified 17 security-level delisting records and confirmed that all have valid daily returns and positive previous-day market caps.
This ensures that these records are not removed by our market-cap weighting procedure.
We retain historical returns, leave post-delisting returns as NaN, and exclude delisted securities from future investment decisions.

In [46]:
#final check
print("Return matrix shape:", return_matrix.shape)
print("Number of companies:", return_matrix.shape[1])
print("Internal missing returns:", internal_missing.sum())
print("Start date:", return_matrix.index.min())
print("End date:", return_matrix.index.max())
print("Maximum daily return:", return_matrix.max().max())
print("Minimum daily return:", return_matrix.min().min())

Return matrix shape: (4024, 100)
Number of companies: 100
Internal missing returns: 0
Start date: 2010-01-04 00:00:00
End date: 2025-12-31 00:00:00
Maximum daily return: 0.359488
Minimum daily return: -0.538647


In [49]:
# Export cleaned data
df.to_csv("2010_2025_Top100_Cleaned.csv", index=False)

# Export company-level daily returns
company_returns.to_csv(
    "2010_2025_Company_Returns.csv", index=False
)

# Export return matrix for portfolio optimisation
return_matrix.to_csv("2010_2025_Return_Matrix.csv")